# flash-ph: GPU-Accelerated Exact Rips Persistent Homology

This notebook demonstrates [flash-ph](https://github.com/ot-triton-lab/flash-ph), a GPU-accelerated library for exact Vietoris-Rips persistent homology (H0, H1, H2) using PyTorch + Triton kernels.

**Requirements:** An NVIDIA A100 GPU runtime. In Colab, go to **Runtime > Change runtime type > A100 GPU**.

**What this notebook covers:**
1. Installation and setup
2. Quick start with `rips_persistence`
3. Threshold selection (`auto_threshold` vs `enclosing_radius`)
4. GUDHI-compatible `RipsComplex` API
5. Visualization (persistence diagrams and barcodes)
6. Scaling benchmarks vs ripser and giotto-ph

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available — select an A100 GPU runtime."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:     {gpu.name}")
print(f"VRAM:    {gpu.total_memory / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
%%capture
!pip install --quiet "flash-ph @ git+https://github.com/ot-triton-lab/flash-ph.git"
!pip install --quiet ripser giotto-ph

In [ ]:
import flash_ph
from flash_ph import rips_persistence, auto_threshold, enclosing_radius, RipsComplex

print(f"flash-ph {flash_ph.__version__}")
print(f"Exports: rips_persistence, auto_threshold, enclosing_radius, RipsComplex")

### JIT Warmup

The first call to `rips_persistence` triggers Triton and Numba JIT compilation (typically 5-15 seconds). Subsequent calls reuse cached kernels and are much faster.

In [ ]:
import time

pts_warmup = torch.randn(100, 3, device="cuda")
t0 = time.perf_counter()
_ = rips_persistence(pts_warmup, max_edge_length=2.0, max_dim=2)
torch.cuda.synchronize()
print(f"Warmup (JIT compile): {time.perf_counter() - t0:.1f}s")

## Quick Start

Generate a random 3D point cloud and compute H0 (connected components) and H1 (loops).

In [ ]:
torch.manual_seed(42)
pts = torch.randn(500, 3, device="cuda")

diagrams = rips_persistence(pts, max_edge_length=1.0, max_dim=1)
h0, h1 = diagrams

print(f"H0 bars: {len(h0)}")
print(f"H1 bars: {len(h1)}")

# Top 5 most persistent H1 features
if len(h1) > 0:
    pers = h1[:, 1] - h1[:, 0]
    pers[h1[:, 1] == float('inf')] = float('inf')
    top5 = pers.argsort(descending=True)[:5]
    print("\nTop 5 H1 bars (birth, death, persistence):")
    for i in top5:
        b, d = h1[i]
        print(f"  ({b:.4f}, {d:.4f}, pers={d - b:.4f})")

## Threshold Selection

flash-ph provides two strategies for choosing `max_edge_length`:

- **`auto_threshold(pts, k, percentile)`** — returns the p-th percentile of k-nearest-neighbor distances. Keeps the Rips graph sparse; recommended for high-dimensional data (d > 3).
- **`enclosing_radius(pts)`** — returns min_x max_y d(x, y). Guarantees no topological features are missed, but may be expensive.

In [ ]:
torch.manual_seed(0)
pts_10d = torch.randn(1000, 10, device="cuda")

thresh_auto = auto_threshold(pts_10d, k=20, percentile=95)
thresh_enc = enclosing_radius(pts_10d)

print(f"auto_threshold (k=20, p=95): {thresh_auto:.3f}")
print(f"enclosing_radius:            {thresh_enc:.3f}")
print(f"Ratio (enclosing / auto):    {thresh_enc / thresh_auto:.1f}x")

# Time each
for name, thr in [("auto_threshold", thresh_auto), ("enclosing_radius", thresh_enc)]:
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = rips_persistence(pts_10d, max_edge_length=thr, max_dim=1)
    torch.cuda.synchronize()
    print(f"  rips_persistence with {name}: {(time.perf_counter() - t0) * 1000:.0f} ms")

## GUDHI-Compatible API

`RipsComplex` provides a GUDHI-style interface backed by the flash-ph GPU pipeline. It accepts the same methods as GUDHI's `RipsComplex`.

In [ ]:
torch.manual_seed(42)
pts_rips = torch.randn(500, 3, device="cuda")

rips = RipsComplex(points=pts_rips, max_edge_length=1.0)
rips.compute_persistence(max_dim=1)

# Betti numbers (count of essential features)
print(f"Betti numbers: {rips.betti_numbers()}")

# Persistent Betti numbers (features alive in [from, to])
print(f"Persistent Betti (0.1, 0.5): {rips.persistent_betti_numbers(0.1, 0.5)}")

# Persistence intervals for H1
h1_intervals = rips.persistence_intervals_in_dimension(1)
print(f"H1 intervals (shape): {h1_intervals.shape}")
print(f"H1 first 3 intervals:\n{h1_intervals[:3]}")

# GUDHI-style persistence() output
pairs = rips.persistence(max_dim=1)
print(f"\nTotal persistence pairs: {len(pairs)}")
print(f"First 5 pairs: {pairs[:5]}")

## Visualization

Persistence diagrams and barcodes are the standard ways to visualize persistent homology.

- **Persistence diagram**: birth vs death scatter plot. Points far from the diagonal represent long-lived (significant) features.
- **Barcode**: horizontal bars sorted by persistence, showing the lifetime of each feature.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

COLORS = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
LABELS = {0: "H0", 1: "H1", 2: "H2"}


def plot_persistence_diagram(diagrams, title="Persistence Diagram", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    max_val = 0
    for dim, diag in enumerate(diagrams):
        d = diag.cpu().numpy()
        finite = d[np.isfinite(d[:, 1])]
        if len(finite) > 0:
            max_val = max(max_val, finite[:, 1].max())
            ax.scatter(finite[:, 0], finite[:, 1], s=15, alpha=0.7,
                       color=COLORS[dim], label=LABELS[dim], zorder=2)
        # Plot essential features as triangles at the top
        essential = d[~np.isfinite(d[:, 1])]
        if len(essential) > 0:
            ax.scatter(essential[:, 0], [max_val * 1.1] * len(essential),
                       s=30, marker="^", color=COLORS[dim], alpha=0.7, zorder=2)
    lim = max_val * 1.15 if max_val > 0 else 1
    ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.5, zorder=1)
    ax.set_xlim(-0.02 * lim, lim)
    ax.set_ylim(-0.02 * lim, lim)
    ax.set_xlabel("Birth")
    ax.set_ylabel("Death")
    ax.set_title(title)
    ax.legend()
    ax.set_aspect("equal")
    return ax


def plot_barcode(diagrams, title="Barcode", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    y = 0
    max_death = 0
    for dim, diag in enumerate(diagrams):
        d = diag.cpu().numpy()
        # Sort by persistence (longest first)
        pers = d[:, 1] - d[:, 0]
        pers[~np.isfinite(pers)] = 1e10
        order = np.argsort(-pers)
        finite_deaths = d[np.isfinite(d[:, 1]), 1]
        if len(finite_deaths) > 0:
            max_death = max(max_death, finite_deaths.max())
        for idx in order:
            b, death = d[idx]
            death_plot = death if np.isfinite(death) else max_death * 1.15
            ax.plot([b, death_plot], [y, y], color=COLORS[dim], lw=1.5, alpha=0.7)
            y += 1
    ax.set_xlabel("Filtration value")
    ax.set_ylabel("Feature")
    ax.set_title(title)
    # Custom legend
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=COLORS[d], lw=2, label=LABELS[d])
               for d in range(len(diagrams))]
    ax.legend(handles=handles)
    ax.set_yticks([])
    return ax


print("Plotting functions defined.")

In [ ]:
# Noisy circle — expect 1 prominent H1 bar
torch.manual_seed(0)
n_circle = 200
theta = torch.linspace(0, 2 * np.pi, n_circle, device="cuda")
circle = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
circle += 0.1 * torch.randn_like(circle)

diags_circle = rips_persistence(circle, max_edge_length=1.0, max_dim=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_persistence_diagram(diags_circle, title="Noisy Circle — Diagram", ax=axes[0])
plot_barcode(diags_circle, title="Noisy Circle — Barcode", ax=axes[1])
fig.tight_layout()
plt.show()

h1_circle = diags_circle[1]
pers_circle = h1_circle[:, 1] - h1_circle[:, 0]
print(f"H1 bars: {len(h1_circle)}, most persistent: {pers_circle.max():.4f}")

In [ ]:
# Clifford torus in R^4 — expect 2 prominent H1 bars
# Embedding: (cos u, sin u, cos v, sin v) — both loops have equal geometry.
# Lives in d=4, so Alpha complexes (limited to d<=3) cannot be used here.
torch.manual_seed(1)
n_torus = 1000
u = torch.rand(n_torus, device="cuda") * 2 * np.pi
v = torch.rand(n_torus, device="cuda") * 2 * np.pi
torus = torch.stack([torch.cos(u), torch.sin(u), torch.cos(v), torch.sin(v)], dim=1)
torus += 0.02 * torch.randn_like(torus)

diags_torus = rips_persistence(torus, max_edge_length=1.5, max_dim=1)

fig, ax = plt.subplots(figsize=(5, 5))
plot_persistence_diagram(diags_torus, title="Clifford Torus in $\\mathbb{R}^4$ — Diagram", ax=ax)
plt.show()

h1_torus = diags_torus[1]
pers_torus = h1_torus[:, 1] - h1_torus[:, 0]
top3 = pers_torus.argsort(descending=True)[:3]
print(f"H1 bars: {len(h1_torus)}")
print(f"Top 3 H1 bars (birth, death, persistence):")
for i in top3:
    b, d = h1_torus[i]
    print(f"  ({b:.4f}, {d:.4f}, pers={d - b:.4f})")

## Scaling Benchmark (d=10, varying n)

We benchmark flash-ph against ripser (CPU, full distance matrix) and giotto-ph (CPU, sparse radius-neighbor input) on Gaussian point clouds in d=10 dimensions, computing H0+H1+H2.

Each configuration is run 3 times (median reported). Thresholds are chosen to match the README benchmarks.

In [ ]:
import statistics


def benchmark_fn(fn, reps=3):
    """Run fn() `reps` times and return median wall-clock time in seconds."""
    times = []
    for _ in range(reps):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


# Import CPU baselines with fallback
try:
    import ripser as _ripser
    def run_ripser(pts_np, thresh):
        _ripser.ripser(pts_np, maxdim=2, thresh=thresh)
    HAS_RIPSER = True
except ImportError:
    HAS_RIPSER = False
    print("ripser not installed — skipping ripser benchmarks.")

try:
    from gph import ripser_parallel
    def run_giotto(pts_np, thresh):
        ripser_parallel(pts_np, maxdim=2, thresh=thresh, n_threads=-1)
    HAS_GIOTTO = True
except ImportError:
    HAS_GIOTTO = False
    print("giotto-ph not installed — skipping giotto-ph benchmarks.")

# Warmup CPU baselines
warmup_np = torch.randn(50, 10).numpy()
if HAS_RIPSER:
    run_ripser(warmup_np, 3.0)
if HAS_GIOTTO:
    run_giotto(warmup_np, 3.0)

print(f"Baselines: ripser={'yes' if HAS_RIPSER else 'no'}, giotto-ph={'yes' if HAS_GIOTTO else 'no'}")

In [ ]:
configs_d10 = [
    (500,   10, 2.8, 2),
    (1000,  10, 2.5, 2),
    (2000,  10, 2.2, 2),
    (5000,  10, 1.8, 2),
    (10000, 10, 1.6, 2),
    (20000, 10, 1.4, 2),
]

results_d10 = []

print(f"{'n':>7} {'flash-ph':>10} {'ripser':>10} {'giotto-ph':>10} {'vs ripser':>10} {'vs giotto':>10}")
print("-" * 63)

for n, d, thresh, max_dim in configs_d10:
    torch.manual_seed(0)
    pts = torch.randn(n, d, device="cuda")
    pts_np = pts.cpu().numpy()

    # flash-ph
    t_flash = benchmark_fn(lambda: rips_persistence(pts, thresh, max_dim=max_dim))

    # ripser
    t_ripser = None
    if HAS_RIPSER:
        try:
            t_ripser = benchmark_fn(lambda: run_ripser(pts_np, thresh), reps=3)
        except (MemoryError, Exception) as e:
            t_ripser = None
            print(f"  (ripser OOM at n={n})")

    # giotto-ph
    t_giotto = None
    if HAS_GIOTTO:
        try:
            t_giotto = benchmark_fn(lambda: run_giotto(pts_np, thresh), reps=3)
        except (MemoryError, Exception) as e:
            t_giotto = None
            print(f"  (giotto-ph OOM at n={n})")

    # Speedups
    sp_ripser = f"{t_ripser / t_flash:.1f}x" if t_ripser else "—"
    sp_giotto = f"{t_giotto / t_flash:.1f}x" if t_giotto else "—"

    print(f"{n:>7} {t_flash*1000:>9.0f}ms"
          f" {t_ripser*1000 if t_ripser else float('nan'):>9.0f}ms"
          f" {t_giotto*1000 if t_giotto else float('nan'):>9.0f}ms"
          f" {sp_ripser:>10} {sp_giotto:>10}")

    results_d10.append({
        "n": n, "d": d, "thresh": thresh,
        "flash": t_flash, "ripser": t_ripser, "giotto": t_giotto,
    })

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ns = [r["n"] for r in results_d10]
t_flash = [r["flash"] * 1000 for r in results_d10]
ax.plot(ns, t_flash, "o-", color="tab:blue", label="flash-ph (A100)", lw=2)

if any(r["ripser"] for r in results_d10):
    ns_r = [r["n"] for r in results_d10 if r["ripser"]]
    t_r = [r["ripser"] * 1000 for r in results_d10 if r["ripser"]]
    ax.plot(ns_r, t_r, "s--", color="tab:red", label="ripser (CPU)", lw=2)

if any(r["giotto"] for r in results_d10):
    ns_g = [r["n"] for r in results_d10 if r["giotto"]]
    t_g = [r["giotto"] * 1000 for r in results_d10 if r["giotto"]]
    ax.plot(ns_g, t_g, "^--", color="tab:green", label="giotto-ph (CPU)", lw=2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Number of points (n)")
ax.set_ylabel("Time (ms)")
ax.set_title("Rips Persistence H0+H1+H2 — d=10 Gaussian")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## High-Dimension Benchmark (d=50)

In very high dimensions, pairwise distances concentrate around their mean, making the Rips graph extremely sparse at moderate thresholds. This is where the GPU pipeline excels — the complex is small, and the GPU overhead is amortized over fast kernel launches.

In [ ]:
configs_d50 = [
    (500,  50, 8.0, 2),
    (1000, 50, 7.5, 2),
]

print(f"{'n':>7} {'d':>3} {'flash-ph':>10} {'ripser':>10} {'giotto-ph':>10} {'vs ripser':>10} {'vs giotto':>10}")
print("-" * 67)

for n, d, thresh, max_dim in configs_d50:
    torch.manual_seed(0)
    pts = torch.randn(n, d, device="cuda")
    pts_np = pts.cpu().numpy()

    t_flash = benchmark_fn(lambda: rips_persistence(pts, thresh, max_dim=max_dim))

    t_ripser = None
    if HAS_RIPSER:
        try:
            t_ripser = benchmark_fn(lambda: run_ripser(pts_np, thresh), reps=3)
        except Exception:
            pass

    t_giotto = None
    if HAS_GIOTTO:
        try:
            t_giotto = benchmark_fn(lambda: run_giotto(pts_np, thresh), reps=3)
        except Exception:
            pass

    sp_ripser = f"{t_ripser / t_flash:.1f}x" if t_ripser else "—"
    sp_giotto = f"{t_giotto / t_flash:.1f}x" if t_giotto else "—"

    print(f"{n:>7} {d:>3} {t_flash*1000:>9.0f}ms"
          f" {t_ripser*1000 if t_ripser else float('nan'):>9.0f}ms"
          f" {t_giotto*1000 if t_giotto else float('nan'):>9.0f}ms"
          f" {sp_ripser:>10} {sp_giotto:>10}")

## Summary

| Feature | flash-ph |
|---------|----------|
| Homology dimensions | H0, H1, H2 |
| Coefficients | Z/2Z |
| Metric | Euclidean |
| Correctness | Exact match with ripser |
| GPU backend | PyTorch + Triton |
| CPU fallback | Numba (cohomology reduction) |
| Target regime | d > 3, sparse Rips graphs |
| Speedup (d=10, n=20K) | ~445x vs ripser, ~79x vs giotto-ph |

**GitHub:** [github.com/ot-triton-lab/flash-ph](https://github.com/ot-triton-lab/flash-ph)

**License:** MIT